In [2]:
# Step 1: Import libraries and define project paths

import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("../data/raw/CMAPSSData")
PROCESSED_DIR = Path("../data/processed")
ARTIFACTS_DIR = Path("../artifacts")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

RUL_CAP = 125
WINDOW_SIZE = 30
RANDOM_STATE = 42

print("Libraries imported successfully!")
print("Data directory:", DATA_DIR.resolve())
print("Processed directory:", PROCESSED_DIR.resolve())
print("RUL cap:", RUL_CAP)
print("Window size:", WINDOW_SIZE)

Libraries imported successfully!
Data directory: /Users/shama/Documents/Predictive_Maintenance_RUL/data/raw/CMAPSSData
Processed directory: /Users/shama/Documents/Predictive_Maintenance_RUL/data/processed
RUL cap: 125
Window size: 30


In [3]:
# Step 2: Load all four C-MAPSS datasets

columns = (
    ["unit", "cycle"]
    + [f"setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

datasets = {}

for fd in ["FD001", "FD002", "FD003", "FD004"]:

    train = pd.read_csv(
        DATA_DIR / f"train_{fd}.txt",
        sep=r"\s+",
        header=None
    )

    test = pd.read_csv(
        DATA_DIR / f"test_{fd}.txt",
        sep=r"\s+",
        header=None
    )

    rul = pd.read_csv(
        DATA_DIR / f"RUL_{fd}.txt",
        sep=r"\s+",
        header=None
    )

    train = train.iloc[:, :26]
    test = test.iloc[:, :26]

    train.columns = columns
    test.columns = columns
    rul.columns = ["RUL"]

    datasets[fd] = {
        "train": train,
        "test": test,
        "rul": rul
    }

    print(
        f"{fd}: Train {train.shape}, "
        f"Test {test.shape}, RUL {rul.shape}"
    )

FD001: Train (20631, 26), Test (13096, 26), RUL (100, 1)
FD002: Train (53759, 26), Test (33991, 26), RUL (259, 1)
FD003: Train (24720, 26), Test (16596, 26), RUL (100, 1)
FD004: Train (61249, 26), Test (41214, 26), RUL (248, 1)


In [4]:
# Step 3: Remove constant sensors separately for each dataset

constant_sensors = {}

for fd, data in datasets.items():

    train = data["train"]
    test = data["test"]

    sensor_columns = [
        col for col in train.columns
        if col.startswith("sensor_")
    ]

    constant = [
        col for col in sensor_columns
        if train[col].nunique() == 1
    ]

    constant_sensors[fd] = constant

    # Remove the same sensors from both train and test
    data["train"] = train.drop(columns=constant)
    data["test"] = test.drop(columns=constant)

    print(f"\n{fd}")
    print("Constant sensors:", constant)
    print("Remaining sensors:", len(sensor_columns) - len(constant))
    print("Updated train shape:", data["train"].shape)
    print("Updated test shape:", data["test"].shape)


FD001
Constant sensors: ['sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
Remaining sensors: 15
Updated train shape: (20631, 20)
Updated test shape: (13096, 20)

FD002
Constant sensors: []
Remaining sensors: 21
Updated train shape: (53759, 26)
Updated test shape: (33991, 26)

FD003
Constant sensors: ['sensor_1', 'sensor_5', 'sensor_16', 'sensor_18', 'sensor_19']
Remaining sensors: 16
Updated train shape: (24720, 21)
Updated test shape: (16596, 21)

FD004
Constant sensors: []
Remaining sensors: 21
Updated train shape: (61249, 26)
Updated test shape: (41214, 26)


In [5]:
# Step 4: Calculate and cap training RUL

for fd, data in datasets.items():

    train = data["train"].copy()

    max_cycles = train.groupby("unit")["cycle"].transform("max")

    train["RUL"] = max_cycles - train["cycle"]

    # Apply RUL cap
    train["RUL"] = train["RUL"].clip(upper=RUL_CAP)

    data["train"] = train

    print(f"\n{fd}")
    print("RUL statistics:")
    print(train["RUL"].describe().round(2))
    print("Maximum RUL after capping:", train["RUL"].max())


FD001
RUL statistics:
count    20631.00
mean        86.83
std         41.67
min          0.00
25%         51.00
50%        103.00
75%        125.00
max        125.00
Name: RUL, dtype: float64
Maximum RUL after capping: 125

FD002
RUL statistics:
count    53759.00
mean        86.91
std         41.67
min          0.00
25%         51.00
50%        103.00
75%        125.00
max        125.00
Name: RUL, dtype: float64
Maximum RUL after capping: 125

FD003
RUL statistics:
count    24720.00
mean        93.14
std         40.63
min          0.00
25%         61.00
50%        123.00
75%        125.00
max        125.00
Name: RUL, dtype: float64
Maximum RUL after capping: 125

FD004
RUL statistics:
count    61249.00
mean        92.99
std         40.67
min          0.00
25%         61.00
50%        122.00
75%        125.00
max        125.00
Name: RUL, dtype: float64
Maximum RUL after capping: 125


In [6]:
# Step 4A: Compare RUL distributions across datasets

rul_summary = []

for fd, data in datasets.items():

    rul_summary.append({
        "Dataset": fd,
        "Mean RUL": data["train"]["RUL"].mean(),
        "Std RUL": data["train"]["RUL"].std(),
        "Minimum RUL": data["train"]["RUL"].min(),
        "Maximum RUL": data["train"]["RUL"].max()
    })

rul_summary_df = pd.DataFrame(rul_summary)

print(rul_summary_df.round(2).to_string(index=False))

Dataset  Mean RUL  Std RUL  Minimum RUL  Maximum RUL
  FD001     86.83    41.67            0          125
  FD002     86.91    41.67            0          125
  FD003     93.14    40.63            0          125
  FD004     92.99    40.67            0          125


In [7]:
# Step 5: Prepare operating settings for clustering

from sklearn.cluster import KMeans

setting_columns = [
    "setting_1",
    "setting_2",
    "setting_3"
]

for fd in ["FD002", "FD004"]:

    train = datasets[fd]["train"]

    print(f"\n{fd}")
    print("Unique setting combinations:", 
          train[setting_columns].drop_duplicates().shape[0])

    print("\nSetting statistics:")
    print(train[setting_columns].describe().round(3))


FD002
Unique setting combinations: 9824

Setting statistics:
       setting_1  setting_2  setting_3
count  53759.000  53759.000  53759.000
mean      23.998      0.572     94.046
std       14.747      0.310     14.238
min        0.000      0.000     60.000
25%       10.005      0.251    100.000
50%       25.001      0.700    100.000
75%       41.998      0.840    100.000
max       42.008      0.842    100.000

FD004
Unique setting combinations: 10232

Setting statistics:
       setting_1  setting_2  setting_3
count  61249.000  61249.000  61249.000
mean      24.000      0.571     94.032
std       14.781      0.311     14.252
min        0.000      0.000     60.000
25%       10.005      0.251    100.000
50%       25.001      0.700    100.000
75%       41.998      0.840    100.000
max       42.008      0.842    100.000


In [8]:
# Step 6: Identify operating regimes using K-Means

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

regime_scalers = {}
regime_models = {}

for fd in ["FD002", "FD004"]:

    train = datasets[fd]["train"]

    # Scale the three operating settings
    scaler = StandardScaler()

    scaled_settings = scaler.fit_transform(
        train[setting_columns]
    )

    # Identify six operating regimes
    kmeans = KMeans(
        n_clusters=6,
        random_state=RANDOM_STATE,
        n_init=10
    )

    regimes = kmeans.fit_predict(scaled_settings)

    # Store regime labels
    train = train.copy()
    train["operating_regime"] = regimes

    datasets[fd]["train"] = train

    regime_scalers[fd] = scaler
    regime_models[fd] = kmeans

    print(f"\n{fd} - Operating regime distribution")
    print(train["operating_regime"].value_counts().sort_index())
    print("Number of identified regimes:", train["operating_regime"].nunique())


FD002 - Operating regime distribution
operating_regime
0    13458
1     8044
2     8122
3     8002
4     8096
5     8037
Name: count, dtype: int64
Number of identified regimes: 6

FD004 - Operating regime distribution
operating_regime
0    15395
1     9224
2     9139
3     9091
4     9238
5     9162
Name: count, dtype: int64
Number of identified regimes: 6


In [9]:
# Step 7: Split engines into training and validation sets

dataset_splits = {}

for fd, data in datasets.items():

    train = data["train"].drop(
        columns=["operating_regime"],
        errors="ignore"
    ).copy()

    test = data["test"].copy()

    engine_ids = train["unit"].unique()

    train_ids, val_ids = train_test_split(
        engine_ids,
        test_size=0.20,
        random_state=RANDOM_STATE
    )

    train_data = train[train["unit"].isin(train_ids)].copy()
    val_data = train[train["unit"].isin(val_ids)].copy()

    dataset_splits[fd] = {
        "train": train_data,
        "val": val_data,
        "test": test,
        "rul": data["rul"].copy()
    }

    print(f"\n{fd}")
    print("Training engines:", train_data["unit"].nunique())
    print("Validation engines:", val_data["unit"].nunique())
    print("Test engines:", test["unit"].nunique())


FD001
Training engines: 80
Validation engines: 20
Test engines: 100

FD002
Training engines: 208
Validation engines: 52
Test engines: 259

FD003
Training engines: 80
Validation engines: 20
Test engines: 100

FD004
Training engines: 199
Validation engines: 50
Test engines: 248


In [10]:
# Step 8: Fit operating regimes using training data only

regime_scalers = {}
regime_models = {}

for fd in ["FD002", "FD004"]:

    train = dataset_splits[fd]["train"].copy()
    val = dataset_splits[fd]["val"].copy()
    test = dataset_splits[fd]["test"].copy()

    # Fit scaler only on training settings
    scaler = StandardScaler()

    train_settings = scaler.fit_transform(
        train[setting_columns]
    )

    # Fit K-Means only on training settings
    kmeans = KMeans(
        n_clusters=6,
        random_state=RANDOM_STATE,
        n_init=10
    )

    train["operating_regime"] = kmeans.fit_predict(
        train_settings
    )

    # Assign regimes to validation and test data
    val_settings = scaler.transform(val[setting_columns])
    test_settings = scaler.transform(test[setting_columns])

    val["operating_regime"] = kmeans.predict(val_settings)
    test["operating_regime"] = kmeans.predict(test_settings)

    # Save fitted objects and updated datasets
    regime_scalers[fd] = scaler
    regime_models[fd] = kmeans

    dataset_splits[fd]["train"] = train
    dataset_splits[fd]["val"] = val
    dataset_splits[fd]["test"] = test

    print(f"\n{fd}")
    print("Training regimes:", train["operating_regime"].nunique())
    print("Validation regimes:", val["operating_regime"].nunique())
    print("Test regimes:", test["operating_regime"].nunique())


FD002
Training regimes: 6
Validation regimes: 6
Test regimes: 6

FD004
Training regimes: 6
Validation regimes: 6
Test regimes: 6


In [11]:
# Step 9: Scale features separately for each dataset

scalers = {}
feature_columns_by_dataset = {}

for fd, splits in dataset_splits.items():

    train = splits["train"].copy()
    val = splits["val"].copy()
    test = splits["test"].copy()

    # Select settings and sensor features
    feature_columns = [
        col for col in train.columns
        if col.startswith("setting_") or col.startswith("sensor_")
    ]

    # Fit scaler only on training data
    scaler = StandardScaler()

    train[feature_columns] = scaler.fit_transform(
        train[feature_columns]
    )

    val[feature_columns] = scaler.transform(
        val[feature_columns]
    )

    test[feature_columns] = scaler.transform(
        test[feature_columns]
    )

    # Store scaler and feature configuration
    scalers[fd] = scaler
    feature_columns_by_dataset[fd] = feature_columns

    # Update datasets
    splits["train"] = train
    splits["val"] = val
    splits["test"] = test

    print(f"\n{fd}")
    print("Number of features:", len(feature_columns))
    print("Training shape:", train.shape)
    print("Validation shape:", val.shape)
    print("Test shape:", test.shape)


FD001
Number of features: 18
Training shape: (16561, 21)
Validation shape: (4070, 21)
Test shape: (13096, 20)

FD002
Number of features: 24
Training shape: (43464, 28)
Validation shape: (10295, 28)
Test shape: (33991, 27)

FD003
Number of features: 19
Training shape: (20012, 22)
Validation shape: (4708, 22)
Test shape: (16596, 21)

FD004
Number of features: 24
Training shape: (49294, 28)
Validation shape: (11955, 28)
Test shape: (41214, 27)


In [12]:
# Step 10: Create sequence windows

def create_windows(data, feature_columns, window_size=30):

    X, y = [], []

    for unit in data["unit"].unique():

        engine_data = (
            data[data["unit"] == unit]
            .sort_values("cycle")
        )

        features = engine_data[feature_columns].values
        targets = engine_data["RUL"].values

        for i in range(len(engine_data) - window_size + 1):

            X.append(features[i:i + window_size])
            y.append(targets[i + window_size - 1])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


window_data = {}

for fd, splits in dataset_splits.items():

    feature_columns = feature_columns_by_dataset[fd]

    X_train, y_train = create_windows(
        splits["train"], feature_columns, WINDOW_SIZE
    )

    X_val, y_val = create_windows(
        splits["val"], feature_columns, WINDOW_SIZE
    )

    window_data[fd] = {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val
    }

    print(f"\n{fd}")
    print("X_train:", X_train.shape)
    print("y_train:", y_train.shape)
    print("X_val:", X_val.shape)
    print("y_val:", y_val.shape)


FD001
X_train: (14241, 30, 18)
y_train: (14241,)
X_val: (3490, 30, 18)
y_val: (3490,)

FD002
X_train: (37432, 30, 24)
y_train: (37432,)
X_val: (8787, 30, 24)
y_val: (8787,)

FD003
X_train: (17692, 30, 19)
y_train: (17692,)
X_val: (4128, 30, 19)
y_val: (4128,)

FD004
X_train: (43523, 30, 24)
y_train: (43523,)
X_val: (10505, 30, 24)
y_val: (10505,)


In [14]:
# Step 11: Create final test sequences with padding

test_data = {}

for fd, splits in dataset_splits.items():

    test = splits["test"]
    feature_columns = feature_columns_by_dataset[fd]
    rul_labels = splits["rul"]["RUL"].to_numpy()

    X_test = []
    padded_engines = []

    engine_ids = sorted(test["unit"].unique())

    for unit in engine_ids:

        engine_data = (
            test[test["unit"] == unit]
            .sort_values("cycle")
        )

        sensor_window = engine_data[feature_columns].values

        # Handle engines with fewer than 30 cycles
        if len(sensor_window) < WINDOW_SIZE:

            missing_rows = WINDOW_SIZE - len(sensor_window)

            padding = np.repeat(
                sensor_window[[0]],
                missing_rows,
                axis=0
            )

            sensor_window = np.vstack([
                padding,
                sensor_window
            ])

            padded_engines.append(unit)

        else:
            sensor_window = sensor_window[-WINDOW_SIZE:]

        X_test.append(sensor_window)

    X_test = np.array(X_test, dtype=np.float32)
    y_test = rul_labels.astype(np.float32)

    if len(X_test) != len(y_test):
        raise ValueError(f"{fd}: Test windows and RUL labels do not match")

    test_data[fd] = {
        "X_test": X_test,
        "y_test": y_test
    }

    print(f"\n{fd}")
    print("X_test:", X_test.shape)
    print("y_test:", y_test.shape)
    print("Padded engines:", len(padded_engines))
    print("Maximum actual test RUL:", y_test.max())


FD001
X_test: (100, 30, 18)
y_test: (100,)
Padded engines: 0
Maximum actual test RUL: 145.0

FD002
X_test: (259, 30, 24)
y_test: (259,)
Padded engines: 6
Maximum actual test RUL: 194.0

FD003
X_test: (100, 30, 19)
y_test: (100,)
Padded engines: 0
Maximum actual test RUL: 145.0

FD004
X_test: (248, 30, 24)
y_test: (248,)
Padded engines: 11
Maximum actual test RUL: 195.0


In [15]:
# Step 12: Validate processed sequence data

for fd in datasets:

    arrays = window_data[fd]
    test_arrays = test_data[fd]

    X_train = arrays["X_train"]
    X_val = arrays["X_val"]
    X_test = test_arrays["X_test"]

    y_train = arrays["y_train"]
    y_val = arrays["y_val"]
    y_test = test_arrays["y_test"]

    print(f"\n{fd}")

    print("Training data valid:", np.isfinite(X_train).all())
    print("Validation data valid:", np.isfinite(X_val).all())
    print("Test data valid:", np.isfinite(X_test).all())

    print("Training targets valid:", np.isfinite(y_train).all())
    print("Validation targets valid:", np.isfinite(y_val).all())
    print("Test targets valid:", np.isfinite(y_test).all())

    assert X_train.shape[1] == WINDOW_SIZE
    assert X_val.shape[1] == WINDOW_SIZE
    assert X_test.shape[1] == WINDOW_SIZE

    assert X_train.shape[2] == X_val.shape[2] == X_test.shape[2]

    assert np.isfinite(X_train).all()
    assert np.isfinite(X_val).all()
    assert np.isfinite(X_test).all()

print("\nAll datasets passed validation!")


FD001
Training data valid: True
Validation data valid: True
Test data valid: True
Training targets valid: True
Validation targets valid: True
Test targets valid: True

FD002
Training data valid: True
Validation data valid: True
Test data valid: True
Training targets valid: True
Validation targets valid: True
Test targets valid: True

FD003
Training data valid: True
Validation data valid: True
Test data valid: True
Training targets valid: True
Validation targets valid: True
Test targets valid: True

FD004
Training data valid: True
Validation data valid: True
Test data valid: True
Training targets valid: True
Validation targets valid: True
Test targets valid: True

All datasets passed validation!


In [16]:
# Step 13: Save processed datasets and preprocessing objects

for fd in datasets:

    # Save sequence data
    np.savez_compressed(
        PROCESSED_DIR / f"{fd}_processed.npz",

        X_train=window_data[fd]["X_train"],
        y_train=window_data[fd]["y_train"],

        X_val=window_data[fd]["X_val"],
        y_val=window_data[fd]["y_val"],

        X_test=test_data[fd]["X_test"],
        y_test=test_data[fd]["y_test"]
    )

    # Save preprocessing configuration
    preprocessing_objects = {
        "scaler": scalers[fd],
        "feature_columns": feature_columns_by_dataset[fd],
        "constant_sensors": constant_sensors[fd],
        "window_size": WINDOW_SIZE,
        "rul_cap": RUL_CAP
    }

    # Include regime objects for FD002 and FD004
    if fd in ["FD002", "FD004"]:
        preprocessing_objects["regime_scaler"] = regime_scalers[fd]
        preprocessing_objects["regime_model"] = regime_models[fd]

    joblib.dump(
        preprocessing_objects,
        ARTIFACTS_DIR / f"{fd}_preprocessing.pkl"
    )

    print(f"{fd}: Processed data and preprocessing objects saved!")

FD001: Processed data and preprocessing objects saved!
FD002: Processed data and preprocessing objects saved!
FD003: Processed data and preprocessing objects saved!
FD004: Processed data and preprocessing objects saved!


In [17]:
# Step 13A: Verify saved files

for fd in datasets:

    data_file = PROCESSED_DIR / f"{fd}_processed.npz"
    config_file = ARTIFACTS_DIR / f"{fd}_preprocessing.pkl"

    print(f"\n{fd}")
    print("Processed data exists:", data_file.exists())
    print("Preprocessing config exists:", config_file.exists())


FD001
Processed data exists: True
Preprocessing config exists: True

FD002
Processed data exists: True
Preprocessing config exists: True

FD003
Processed data exists: True
Preprocessing config exists: True

FD004
Processed data exists: True
Preprocessing config exists: True
